# CellIQ CT nodule detector: can we make it better?

The open-source MONAI `lung_nodule_ct_detection` bundle finds nodule **candidates**. This notebook measures three cheap ways to improve how useful its list is, on a larger set of public LUNA16 scans than before:

1. **More scans.** Run on up to 80 scans (the first evaluation used 24, which gave wide ranges).
2. **Re-rank.** Keep only the top few candidates per scan, or re-order them by a learned score.
3. **Second-stage filter.** Train a small model on each candidate (its score, size and the CT values around it) to tell real nodules from false alarms. It is trained and tested with **cross-validation by scan**, so a scan is never in both the training and test part.

Everything is judged the same way as before: **share of radiologist-marked nodules found at a given number of false alarms per scan** (FROC), with a bootstrap range. If a change does not help, the notebook says so.

It saves every raw detection (`detections.csv`) so any later analysis can be done without rerunning the detector.

**How to run:** Runtime > Change runtime type > T4 GPU, then Runtime > Run all. About 45 to 75 minutes.

A candidate is for a radiologist to look at. It is not a cancer diagnosis. Credit: LUNA16 (CC BY 4.0), built from LIDC-IDRI (CC BY 3.0). Research use only.

## 1. Settings

In [ ]:
import os, sys, json, time, subprocess, shutil

N_SCANS = 80          # how many scans to test (subset0 has 89; only those with a raw file <= MAX_RAW_MB are eligible)
MAX_RAW_MB = 120      # only scans whose compressed raw file is at most this big (keeps downloads quick)
SEED = 0              # scan selection seed (same seed = same scans)
BOOT_N = 500          # bootstrap repeats for the confidence interval
BOOT_SEED = 1
FETCH_THREADS = 4

BUNDLE = "/content/bundles/lung_nodule_ct_detection"
LUNA_DIR = "/content/luna"
CASE_DIR = "/content/cases"
DET_DIR = "/content/det_out"
OUT_DIR = "/content/improve_out"
N_REPEATS = 3        # repeats of the 5-fold cross-validation (different scan shuffles)
ZENODO_RECORDS = [3723295, 2595812, 2595813, 2596478, 4121926]   # LUNA16 records on Zenodo


def run(cmd, **kw):
    """Run a command, return the finished process, and show the tail of its output if it failed."""
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode:
        print("COMMAND FAILED:", " ".join(map(str, cmd))[:200])
        print((r.stdout + r.stderr)[-1500:])
    return r


for d in (LUNA_DIR, CASE_DIR, DET_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)
print("settings ok")

## 2. Install the libraries and download the detector bundle

In [ ]:
r = run([sys.executable, "-m", "pip", "install", "-q", "monai[itk,nibabel,tqdm]", "fire", "pytorch-ignite",
         "SimpleITK", "pandas", "remotezip", "matplotlib"])
assert r.returncode == 0, "pip install failed (see above)"
import monai, torch
print("monai", monai.__version__, "| GPU available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU. Switch the runtime to T4 GPU or inference will be very slow.")

r = run([sys.executable, "-m", "monai.bundle", "download", "lung_nodule_ct_detection", "--bundle_dir", "/content/bundles"])
assert r.returncode == 0 and os.path.isdir(BUNDLE), "bundle download failed (see above)"
print("bundle ready:", BUNDLE)

## 3. Find the LUNA16 files on Zenodo

In [ ]:
import requests

WANT = {"subset0.zip": None, "annotations.csv": None, "annotations_excluded.csv": None}
for rid in ZENODO_RECORDS:
    try:
        rec = requests.get(f"https://zenodo.org/api/records/{rid}", timeout=30).json()
    except Exception as e:
        print("record", rid, "lookup failed:", type(e).__name__)
        continue
    for f in rec.get("files", []):
        key = f.get("key") or f.get("filename")
        links = f.get("links") or {}
        if key in WANT and WANT[key] is None:
            WANT[key] = {"record": rid, "url": links.get("self") or links.get("download"), "size": f.get("size")}
print({k: (v["record"], v["size"]) if v else None for k, v in WANT.items()})
assert WANT["subset0.zip"] and WANT["annotations.csv"], "subset0.zip or annotations.csv was not found on Zenodo"

for key in ("annotations.csv", "annotations_excluded.csv"):
    if WANT[key]:
        run(["curl", "-L", "-s", "--retry", "3", "-o", f"{LUNA_DIR}/{key}", WANT[key]["url"]])
        print(key, os.path.getsize(f"{LUNA_DIR}/{key}"), "bytes")
SUBSET_URL = WANT["subset0.zip"]["url"]

## 4. Pick scans at random and fetch only those files

In [ ]:
import numpy as np, pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from remotezip import RemoteZip

ann = pd.read_csv(f"{LUNA_DIR}/annotations.csv")
excl_path = f"{LUNA_DIR}/annotations_excluded.csv"
excl = pd.read_csv(excl_path) if os.path.exists(excl_path) else None

with RemoteZip(SUBSET_URL) as rz:
    infos = {i.filename: i for i in rz.infolist()}
mhd = sorted(n for n in infos if n.endswith(".mhd"))
eligible = [n for n in mhd if n[:-4] + ".raw" in infos and infos[n[:-4] + ".raw"].compress_size <= MAX_RAW_MB * 1e6]
rng = np.random.RandomState(SEED)
chosen = sorted(rng.choice(eligible, size=min(N_SCANS, len(eligible)), replace=False).tolist())
print(f"scans in subset0: {len(mhd)} | eligible: {len(eligible)} | chosen: {len(chosen)}")
print("compressed data to fetch: %.0f MB" % (sum(infos[n[:-4] + ".raw"].compress_size for n in chosen) / 1e6))


def fetch(name):
    for attempt in range(3):
        try:
            with RemoteZip(SUBSET_URL) as z:
                z.extract(name, CASE_DIR)
                z.extract(name[:-4] + ".raw", CASE_DIR)
            return name, True
        except Exception:
            time.sleep(3)
    return name, False


t0, done = time.time(), 0
with ThreadPoolExecutor(max_workers=FETCH_THREADS) as ex:
    for fut in as_completed([ex.submit(fetch, n) for n in chosen]):
        name, ok = fut.result()
        done += 1
        print(f"{done}/{len(chosen)} {'ok' if ok else 'FAILED'} {os.path.basename(name)[-12:-4]} | {time.time() - t0:.0f}s", flush=True)

scans = [n for n in chosen if os.path.exists(f"{CASE_DIR}/{n[:-4]}.raw") and os.path.exists(f"{CASE_DIR}/{n}")]
assert scans, "no scans were downloaded"
uids = [os.path.basename(n)[:-4] for n in scans]
json.dump({"validation": [{"image": n} for n in scans]}, open("/content/datalist.json", "w"))
print("ready:", len(scans), "scans | with marked nodules:", sum(u in set(ann.seriesuid) for u in uids),
      "| marked nodules:", int(ann.seriesuid.isin(uids).sum()))

## 5. Run the detector on every chosen scan

In [ ]:
cfg = json.load(open(f"{BUNDLE}/configs/inference.json"))
cfg["whether_raw_luna16"] = True                 # the scans are raw LUNA16 volumes
cfg["dataset_dir"] = CASE_DIR
cfg["data_list_file_path"] = "/content/datalist.json"
cfg["output_dir"] = DET_DIR
cfg["output_filename"] = "result.json"
cfg["dataloader"]["num_workers"] = 0             # more workers ran out of RAM on Colab
json.dump(cfg, open("/content/infer_cfg.json", "w"), indent=1)

t0 = time.time()
r = run([sys.executable, "-m", "monai.bundle", "run", "--config_file", "/content/infer_cfg.json",
         "--meta_file", f"{BUNDLE}/configs/metadata.json"], cwd=BUNDLE)
print(f"inference exit code: {r.returncode} | {time.time() - t0:.0f}s")
assert r.returncode == 0, "inference failed (see above)"

raw = json.load(open(f"{DET_DIR}/result.json"))
records = raw if isinstance(raw, list) else raw.get("validation", [raw])
assert len(records) == len(scans), f"expected {len(scans)} results, got {len(records)}"
print("results:", len(records), "for", len(scans), "scans")

## 6. Match detections to the radiologists' marks (same rule as before)

In [ ]:
def record_matches(rec, uid):
    """True unless the record names a different scan (records without a file name are matched by order)."""
    img = str(rec.get("image", "")) if isinstance(rec, dict) else ""
    return (not img) or (uid in img)


assert all(record_matches(rec, uids[i]) for i, rec in enumerate(records)), "result order does not match the scan list"

per_scan = []
for i, rec in enumerate(records):
    uid = uids[i]
    boxes = np.array(rec["box"], dtype=float).reshape(-1, 6)           # [cx, cy, cz, w, h, d] in LUNA16 world mm
    scores = np.array(rec["label_scores"], dtype=float).reshape(-1)
    centres = boxes[:, :3]                                              # already world coordinates: no flip needed
    a = ann[ann.seriesuid == uid]
    marks = a[["coordX", "coordY", "coordZ"]].values
    radii = a["diameter_mm"].values / 2.0
    ignorable = excl[excl.seriesuid == uid][["coordX", "coordY", "coordZ"]].values if excl is not None else np.zeros((0, 3))

    best = np.zeros(len(marks))              # highest score of any detection that hits each marked nodule
    false_scores = []
    for c, s in zip(centres, scores):
        hit = np.linalg.norm(marks - c, axis=1) <= radii if len(marks) else np.zeros(0, bool)
        if hit.any():
            best[hit] = np.maximum(best[hit], s)
        elif len(ignorable) and (np.linalg.norm(ignorable - c, axis=1) <= 5.0).any():
            continue
        else:
            false_scores.append(s)
    per_scan.append({"uid": uid, "marked": len(marks), "best": best, "false": np.array(false_scores)})

n_scans = len(per_scan)
n_marked = int(sum(p["marked"] for p in per_scan))
n_with = int(sum(p["marked"] > 0 for p in per_scan))
print(f"scans: {n_scans} | scans with marked nodules: {n_with} | marked nodules: {n_marked}")

## 7. One row per detection

Each detection gets: whether it hits a marked nodule, whether it lies near an "excluded" finding (those are ignored, as before), and simple box features.

In [ ]:
import pandas as pd

rows = []
marked = []
for i, rec in enumerate(records):
    uid = uids[i]
    boxes = np.array(rec["box"], dtype=float).reshape(-1, 6)
    scores = np.array(rec["label_scores"], dtype=float).reshape(-1)
    a = ann[ann.seriesuid == uid]
    marks = a[["coordX", "coordY", "coordZ"]].values
    radii = a["diameter_mm"].values / 2.0
    ign = excl[excl.seriesuid == uid][["coordX", "coordY", "coordZ"]].values if excl is not None else np.zeros((0, 3))
    marked.append(len(marks))
    for b, s in zip(boxes, scores):
        c = b[:3]
        hit = np.where(np.linalg.norm(marks - c, axis=1) <= radii)[0] if len(marks) else np.zeros(0, int)
        ignorable = len(hit) == 0 and len(ign) > 0 and bool((np.linalg.norm(ign - c, axis=1) <= 5.0).any())
        rows.append(dict(scan=i, uid=uid, cx=c[0], cy=c[1], cz=c[2], w=b[3], h=b[4], d=b[5], score=float(s),
                         hit=bool(len(hit)), hit_ids=[int(m) for m in hit], ignorable=ignorable))
dets = pd.DataFrame(rows)
marked = np.array(marked)
n_scans = len(uids)
groups = [np.where(dets.scan.values == i)[0] for i in range(n_scans)]
print(f"scans {n_scans} | detections {len(dets)} | marked nodules {int(marked.sum())} | detections that hit a nodule {int(dets.hit.sum())} | ignorable {int(dets.ignorable.sum())}")
print("detections per scan: median", int(np.median([len(g) for g in groups])), "| max", max(len(g) for g in groups))

## 8. Box and score features, and CT-value features around each candidate

CT-value features come from a 32 mm cube around the candidate: average and spread of the Hounsfield values, how solid the centre is compared with its surroundings, and how much of the cube is air.

In [ ]:
import SimpleITK as sitk

# box / score features
dets["d_mean"] = dets[["w", "h", "d"]].mean(axis=1)
dets["d_min"] = dets[["w", "h", "d"]].min(axis=1)
dets["d_max"] = dets[["w", "h", "d"]].max(axis=1)
dets["aspect"] = dets.d_max / dets.d_min.clip(lower=0.5)
dets["volume"] = dets.w * dets.h * dets.d
dets["rank_in_scan"] = dets.groupby("scan").score.rank(ascending=False, method="first")
dets["score_vs_max"] = dets.score / dets.groupby("scan").score.transform("max").clip(lower=1e-6)
dets["n_dets_scan"] = dets.groupby("scan").score.transform("size")
dets["n_strong_scan"] = dets.groupby("scan").score.transform(lambda s: (s >= 0.5).sum())

HALF_MM, CORE_MM = 16.0, 6.0
pix_cols = ["hu_mean", "hu_std", "hu_p10", "hu_p90", "hu_min", "hu_max", "core_mean", "core_std", "core_p90", "shell_mean",
            "core_minus_shell", "core_solid", "core_dense", "cube_air", "z_rel"]
pix = np.full((len(dets), len(pix_cols)), np.nan, dtype=np.float64)
t0 = time.time()
for i, name in enumerate(scans):
    img = sitk.ReadImage(f"{CASE_DIR}/{name}")
    arr = sitk.GetArrayFromImage(img).astype(np.float32)                  # z, y, x
    sp = np.array(img.GetSpacing(), dtype=float)                          # x, y, z (mm)
    size = np.array(img.GetSize(), dtype=int)
    zrange = (img.TransformIndexToPhysicalPoint([0, 0, 0])[2], img.TransformIndexToPhysicalPoint([int(size[0]) - 1, int(size[1]) - 1, int(size[2]) - 1])[2])
    for k in groups[i]:
        c = dets.loc[k, ["cx", "cy", "cz"]].values.astype(float)
        ix, iy, iz = [float(v) for v in img.TransformPhysicalPointToContinuousIndex(c.tolist())]
        half = np.ceil(HALF_MM / sp).astype(int)
        x0, x1 = int(round(ix)) - half[0], int(round(ix)) + half[0] + 1
        y0, y1 = int(round(iy)) - half[1], int(round(iy)) + half[1] + 1
        z0, z1 = int(round(iz)) - half[2], int(round(iz)) + half[2] + 1
        cube = np.full((z1 - z0, y1 - y0, x1 - x0), -1000.0, dtype=np.float32)          # outside the scan counts as air
        sx0, sx1, sy0, sy1, sz0, sz1 = max(x0, 0), min(x1, size[0]), max(y0, 0), min(y1, size[1]), max(z0, 0), min(z1, size[2])
        if sx1 > sx0 and sy1 > sy0 and sz1 > sz0:
            cube[sz0 - z0:sz1 - z0, sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0] = arr[sz0:sz1, sy0:sy1, sx0:sx1]
        zz, yy, xx = np.meshgrid((np.arange(z0, z1) - iz) * sp[2], (np.arange(y0, y1) - iy) * sp[1], (np.arange(x0, x1) - ix) * sp[0], indexing="ij")
        rr = np.sqrt(xx ** 2 + yy ** 2 + zz ** 2)
        core, shell = cube[rr <= CORE_MM], cube[(rr > 2 * CORE_MM / 1.0) & (rr <= HALF_MM)]
        if len(core) == 0 or len(shell) == 0:
            continue
        flat = cube.ravel()
        pix[k] = [flat.mean(), flat.std(), np.percentile(flat, 10), np.percentile(flat, 90), flat.min(), flat.max(),
                  core.mean(), core.std(), np.percentile(core, 90), shell.mean(), core.mean() - shell.mean(),
                  float((core > -300).mean()), float((core > 0).mean()), float((flat < -800).mean()),
                  (c[2] - zrange[0]) / (zrange[1] - zrange[0] + 1e-6)]
    print(f"features {i + 1}/{n_scans} | {time.time() - t0:.0f}s", flush=True) if (i + 1) % 10 == 0 else None
for j, col in enumerate(pix_cols):
    dets[col] = pix[:, j]
dets[pix_cols] = dets[pix_cols].fillna(dets[pix_cols].median())
box_cols = ["score", "d_mean", "d_min", "d_max", "aspect", "volume", "rank_in_scan", "score_vs_max", "n_dets_scan", "n_strong_scan"]
print("feature table:", dets.shape, "| missing CT features before fill:", int(np.isnan(pix).any(axis=1).sum()))

## 9. FROC tools, and the baseline (the detector's own score)

`build_per_scan(scores)` turns any scoring of the detections into the same found/false-alarm bookkeeping used before, so every variant is judged by the identical rule.

In [ ]:
TARGETS = [0.125, 0.25, 0.5, 1, 2, 4, 8]
hit_arr, ign_arr = dets.hit.values, dets.ignorable.values
hit_ids = dets.hit_ids.values


def build_per_scan(sc):
    res = []
    for i, g in enumerate(groups):
        best = np.zeros(marked[i])
        fs = []
        for k in g:
            if hit_arr[k]:
                for m in hit_ids[k]:
                    best[m] = max(best[m], sc[k])
            elif not ign_arr[k]:
                fs.append(sc[k])
        res.append((best, np.array(fs, dtype=float)))
    return res


def froc_points(per, ids):
    """Sensitivity and false alarms per scan at every distinct score, for the given (possibly repeated) scans."""
    best = np.concatenate([per[i][0] for i in ids]) if len(ids) else np.zeros(0)
    false = np.concatenate([per[i][1] for i in ids]) if len(ids) else np.zeros(0)
    thr = np.unique(np.concatenate([best[best > 0], false, [0.0]]))
    thr = thr[thr > 0] if (thr > 0).any() else np.array([0.5])
    sb, sf = np.sort(best), np.sort(false)
    sens = 1.0 - np.searchsorted(sb, thr, side="left") / max(len(best), 1)
    fpps = (len(sf) - np.searchsorted(sf, thr, side="left")) / max(len(ids), 1)
    return sens, fpps


def sens_at(sens, fpps, target):
    ok = sens[fpps <= target]
    return float(ok.max()) if len(ok) else 0.0


def summarize(per, ids=None):
    ids = list(range(n_scans)) if ids is None else ids
    s, f = froc_points(per, ids)
    tab = {str(t): round(sens_at(s, f, t), 3) for t in TARGETS}
    return tab, float(np.mean([sens_at(s, f, t) for t in TARGETS])), (s, f)


VARIANTS = {}
VARIANTS["baseline (detector score)"] = dets.score.values.copy()
tab, cpm, _ = summarize(build_per_scan(VARIANTS["baseline (detector score)"]))
print("baseline: share of marked nodules found at most this many false alarms per scan:", tab, "| CPM %.3f" % cpm)
print("marked nodules:", int(marked.sum()), "in", int((marked > 0).sum()), "scans with nodules, out of", n_scans, "scans")

## 10. Idea A: keep only the top few candidates per scan

No training: simply drop every candidate ranked below K in its scan.

In [ ]:
for K in (1, 2, 3, 5, 8):
    VARIANTS[f"top {K} per scan"] = np.where(dets.rank_in_scan.values <= K, dets.score.values, -1.0)
    tab, cpm, (s, f) = summarize(build_per_scan(VARIANTS[f"top {K} per scan"]))
    keep = dets.rank_in_scan.values <= K
    print(f"top {K}: keeps {keep.sum() / n_scans:.2f} candidates per scan | share of marked nodules still found: "
          f"{sens_at(s, f, 1e9):.3f} | at 1 false alarm per scan: {tab['1']}")

## 11. Idea B: a second-stage filter, tested by cross-validation on scans

Three small models learn "is this candidate a real marked nodule?":
- **size model**: the detector's score, box size, and rank within the scan.
- **CT model**: the same plus the CT-value features around the candidate.
- **linear CT model**: the same features with a plain logistic regression (the safest choice when there are few real nodules).

Each model is trained on 4/5 of the **scans** and scores the other 1/5, repeated until every scan has been scored by a model that never saw it. This is repeated with different shuffles and the scores averaged. Candidates near "excluded" findings are left out of training and testing, as before.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

usable = ~ign_arr
y_all = hit_arr.astype(int)
print("training rows:", int(usable.sum()), "| real nodule candidates among them:", int(y_all[usable].sum()))


def make_model(kind):
    if kind == "gbm":
        return GradientBoostingClassifier(n_estimators=150, max_depth=2, learning_rate=0.05, subsample=0.8, random_state=0)
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.3, max_iter=2000, class_weight="balanced"))


def cv_scores(cols, kind):
    X = dets[cols].values.astype(float)
    acc = np.zeros(len(dets))
    for rep in range(N_REPEATS):
        fold_of_scan = np.random.RandomState(rep).permutation(n_scans) % 5
        fold = fold_of_scan[dets.scan.values]
        for f in range(5):
            tr = usable & (fold != f)
            te = fold == f
            if y_all[tr].sum() < 3:
                continue
            m = make_model(kind).fit(X[tr], y_all[tr])
            acc[te] += m.predict_proba(X[te])[:, 1] / N_REPEATS
    return acc


MODEL_SPECS = {
    "size model (GBM)": (box_cols, "gbm"),
    "CT model (GBM)": (box_cols + pix_cols, "gbm"),
    "linear CT model": (box_cols + pix_cols, "lin"),
}
for name, (cols, kind) in MODEL_SPECS.items():
    VARIANTS[name] = cv_scores(cols, kind)
    tab, cpm, _ = summarize(build_per_scan(VARIANTS[name]))
    print(f"{name:20s} found at 0.5 / 1 / 2 / 4 false alarms per scan: {tab['0.5']} / {tab['1']} / {tab['2']} / {tab['4']} | CPM {cpm:.3f}")

## 12. Compare every variant with the baseline, with a bootstrap range

Scans are resampled with replacement 500 times. For each resample we compute the share found at 1 false alarm per scan and the CPM for every variant, and the **difference from the baseline**. If the 95% range of the difference includes zero, the change is not proven to help.

In [ ]:
pers = {name: build_per_scan(sc_) for name, sc_ in VARIANTS.items()}
brng = np.random.RandomState(BOOT_SEED)
boot_ids = [brng.randint(0, n_scans, n_scans) for _ in range(BOOT_N)]
BASE = "baseline (detector score)"
store = {name: {"s1": [], "cpm": []} for name in VARIANTS}
for ids in boot_ids:
    for name, per in pers.items():
        s_, f_ = froc_points(per, ids)
        store[name]["s1"].append(sens_at(s_, f_, 1))
        store[name]["cpm"].append(np.mean([sens_at(s_, f_, t) for t in TARGETS]))

results = {}
for name, per in pers.items():
    tab, cpm, (s_, f_) = summarize(per)
    d1 = np.array(store[name]["s1"]) - np.array(store[BASE]["s1"])
    dc = np.array(store[name]["cpm"]) - np.array(store[BASE]["cpm"])
    lo1, hi1 = np.percentile(d1, [2.5, 97.5])
    loc, hic = np.percentile(dc, [2.5, 97.5])
    results[name] = {"sensitivity_at_false_alarms_per_scan": tab, "cpm": round(cpm, 3),
                     "sens_at_1_95_range": [round(float(np.percentile(store[name]["s1"], 2.5)), 3), round(float(np.percentile(store[name]["s1"], 97.5)), 3)],
                     "diff_vs_baseline_sens_at_1": {"mean": round(float(d1.mean()), 3), "range95": [round(float(lo1), 3), round(float(hi1), 3)]},
                     "diff_vs_baseline_cpm": {"mean": round(float(dc.mean()), 3), "range95": [round(float(loc), 3), round(float(hic), 3)]},
                     "clearly_better_than_baseline": bool(lo1 > 0 or loc > 0),
                     "clearly_worse_than_baseline": bool(hi1 < 0 or hic < 0)}
    print(f"{name:26s} @1/scan {tab['1']:.3f} (range {results[name]['sens_at_1_95_range']}) | CPM {cpm:.3f} | "
          f"diff at 1/scan {d1.mean():+.3f} {[round(float(lo1), 3), round(float(hi1), 3)]} | diff CPM {dc.mean():+.3f} {[round(float(loc), 3), round(float(hic), 3)]}"
          f"{'  <-- clearly better' if results[name]['clearly_better_than_baseline'] else ''}{'  <-- clearly worse' if results[name]['clearly_worse_than_baseline'] else ''}")

## 13. Curves, saved files and download zip

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.5, 4.6))
for name in (BASE, "size model (GBM)", "CT model (GBM)", "linear CT model"):
    s_, f_ = summarize(pers[name])[2]
    o = np.argsort(f_, kind="stable")
    ax.step(f_[o], s_[o], where="post", lw=2 if name == BASE else 1.5, label=name)
ax.set_xscale("log"); ax.set_xlim(0.1, 10); ax.set_ylim(0, 1.02)
ax.set_xlabel("False alarms per scan"); ax.set_ylabel("Share of marked nodules found")
ax.set_title(f"FROC: {n_scans} LUNA16 scans, {int(marked.sum())} marked nodules")
ax.grid(alpha=0.3); ax.legend(loc="lower right", fontsize=8)
plt.tight_layout(); os.makedirs(OUT_DIR, exist_ok=True); plt.savefig(f"{OUT_DIR}/froc_variants.png", dpi=140); plt.close(fig)

out_df = dets.drop(columns=["hit_ids"]).copy()
out_df["hit_ids"] = dets.hit_ids.apply(lambda v: ";".join(map(str, v)))
for name in ("size model (GBM)", "CT model (GBM)", "linear CT model"):
    out_df["cv_" + name.replace(" ", "_").replace("(", "").replace(")", "")] = VARIANTS[name]
out_df.to_csv(f"{OUT_DIR}/detections.csv", index=False)

# final second-stage models trained on all scans (for use on new scans; their cross-validated numbers are above)
import joblib
X_all = dets[box_cols + pix_cols].values.astype(float)
final = make_model("gbm").fit(X_all[usable], y_all[usable])
joblib.dump({"model": final, "features": box_cols + pix_cols, "kind": "gbm", "note": "trained on all scans in this run; judge it by the cross-validated numbers, not by its fit"},
            f"{OUT_DIR}/ct_rescorer_gbm.joblib")
final_lin = make_model("lin").fit(X_all[usable], y_all[usable])
joblib.dump({"model": final_lin, "features": box_cols + pix_cols, "kind": "lin", "note": "trained on all scans in this run"},
            f"{OUT_DIR}/ct_rescorer_linear.joblib")

summary = {
    "model": "MONAI lung_nodule_ct_detection v0.6.9 (Apache-2.0)",
    "data": "LUNA16 subset0 (CC BY 4.0) / LIDC-IDRI (CC BY 3.0)",
    "n_scans": int(n_scans), "n_scans_with_nodules": int((marked > 0).sum()), "n_marked_nodules": int(marked.sum()),
    "n_detections": int(len(dets)), "n_detections_that_hit": int(dets.hit.sum()),
    "selection": f"random sample (seed {SEED}) of subset0 scans whose raw file is at most {MAX_RAW_MB} MB compressed",
    "validation": f"{N_REPEATS} x 5-fold cross-validation by scan for the second-stage models; bootstrap over scans ({BOOT_N} resamples)",
    "rule": "a marked nodule counts as found if a detection's centre lies within the nodule's radius (LUNA16 rule); detections near annotations_excluded findings are ignored",
    "variants": results,
    "note": "Small sample (see n_marked_nodules): read the ranges. Second-stage models were never tested on scans they were trained on.",
}
json.dump(summary, open(f"{OUT_DIR}/improve_summary.json", "w"), indent=1)
shutil.make_archive("/content/cellIQ_ct_improve", "zip", OUT_DIR)
print("zip ready:", os.path.getsize("/content/cellIQ_ct_improve.zip"), "bytes")
from IPython.display import Image, display
display(Image(f"{OUT_DIR}/froc_variants.png"))
print("Download /content/cellIQ_ct_improve.zip from the file panel on the left.")